# VALIDATION 1: coupled vs uncoupled signal

In [1]:

import numpy as np
from pactools import Comodulogram

# ============================================================
# SETTINGS
# ============================================================

fs = 500
n_epochs = 84
n_times = 249
n_surrogates = 99

theta_freq = 6
gamma_freq = 50

theta_range = np.linspace(4, 8, 20)
gamma_range = np.linspace(30, 80, 20)

rng = np.random.default_rng(42)

# ============================================================
# GENERATE SIMULATED DATA
# ============================================================

def simulate_pac(coupled=False):

    epochs = []

    t = np.arange(n_times) / fs

    for epoch in range(n_epochs):

        # Random phases for each trial
        theta_phase = rng.uniform(0, 2 * np.pi)
        gamma_phase = rng.uniform(0, 2 * np.pi)

        theta_angle = 2 * np.pi * theta_freq * t + theta_phase

        theta = np.sin(theta_angle)

        # Gamma amplitude
        if coupled:
            gamma_amplitude = 1 + 0.8 * np.cos(theta_angle)
        else:
            gamma_amplitude = np.ones(n_times)

        gamma = gamma_amplitude * np.sin(
            2 * np.pi * gamma_freq * t + gamma_phase
        )

        noise = 0.2 * rng.standard_normal(n_times)

        signal = theta + 0.5 * gamma + noise

        epochs.append(signal)

    return np.stack(epochs)


# Independent theta and gamma
data_uncoupled = simulate_pac(coupled=False)

# Known theta-gamma coupling
data_coupled = simulate_pac(coupled=True)

print("Uncoupled shape:", data_uncoupled.shape)
print("Coupled shape:", data_coupled.shape)


Uncoupled shape: (84, 249)
Coupled shape: (84, 249)


In [2]:

# ============================================================
# PAC AND SURROGATE ANALYSIS
# ============================================================

def calculate_surrogate_pac(data):

    data_flat = data.reshape(1, -1)

    estimator = Comodulogram(
        fs=fs,
        low_fq_range=theta_range,
        high_fq_range=gamma_range,
        method="tort",
        n_surrogates=n_surrogates,
        minimum_shift=1.0,
        random_state=42,
        progress_bar=False
    )

    estimator.fit(data_flat, data_flat)

    # Average across frequency pairs
    pac_original = np.mean(estimator.comod_)

    pac_surrogates = np.mean(
        estimator.surrogates_,
        axis=(1, 2)
    )

    # Empirical one-sided p-value
    p_value = (
        1 + np.sum(pac_surrogates >= pac_original)
    ) / (n_surrogates + 1)

    return pac_original, pac_surrogates, p_value


# Uncoupled data
pac_unc, sur_unc, p_unc = calculate_surrogate_pac(
    data_uncoupled
)

# Coupled data
pac_coup, sur_coup, p_coup = calculate_surrogate_pac(
    data_coupled
)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\nUNCOUPLED SIGNAL")
print("Original PAC:", pac_unc)
print("Mean surrogate:", sur_unc.mean())
print("Surrogate SD:", sur_unc.std())
print("Empirical p:", p_unc)

print("\nCOUPLED SIGNAL")
print("Original PAC:", pac_coup)
print("Mean surrogate:", sur_coup.mean())
print("Surrogate SD:", sur_coup.std())
print("Empirical p:", p_coup)



UNCOUPLED SIGNAL
Original PAC: 7.859749195209811e-05
Mean surrogate: 9.943773324600449e-05
Surrogate SD: 2.6151570523442224e-05
Empirical p: 0.81

COUPLED SIGNAL
Original PAC: 0.02117661256707229
Mean surrogate: 0.0002271443578434485
Surrogate SD: 0.00010330014814936564
Empirical p: 0.01


Conclusion:
The implementation distinguished a strongly coupled synthetic signal from an uncoupled one.

# VALIDATION 2: event-locked theta and gamma activity without genuine phase–amplitude coupling

In [4]:

# ============================================================
# SIMULATION 3: EVENT-LOCKED ACTIVITY WITHOUT DIRECT PAC
# ============================================================

rng = np.random.default_rng(123)

t = np.arange(n_times) / fs

event_locked_epochs = []

for epoch in range(n_epochs):

    # Theta: same phase at event onset (phase reset)
    theta = np.sin(2 * np.pi * 6 * t)

    # Gamma: burst at 250 ms after the event
    # Amplitude depends on time, not explicitly on theta phase
    gamma_envelope = 0.2 + 1.5 * np.exp(
        -0.5 * ((t - 0.25) / 0.05) ** 2
    )

    # Random gamma carrier phase
    gamma_phase = rng.uniform(0, 2 * np.pi)

    gamma = gamma_envelope * np.sin(
        2 * np.pi * 50 * t + gamma_phase
    )

    noise = 0.2 * rng.standard_normal(n_times)

    signal = theta + 0.5 * gamma + noise

    event_locked_epochs.append(signal)

data_event_locked = np.stack(event_locked_epochs)

print("Event-locked data shape:", data_event_locked.shape)


### Step 2. Apply our existing surrogate procedure
pac_event, sur_event, p_event = calculate_surrogate_pac(
    data_event_locked
)

print("\nEVENT-LOCKED SIGNAL")
print("Original PAC:", pac_event)
print("Mean surrogate:", sur_event.mean())
print("Surrogate SD:", sur_event.std())
print("Empirical p:", p_event)

### Step 3. How will we interpret the result?

# There are two main possibilities.

# | Result | Interpretation |
# |---|---|
# | p > 0.05 | No significant PAC detected in this particular event-locked simulation |
# | p ≤ 0.05 | The procedure detects PAC even though no direct theta–gamma modulation was explicitly introduced |

# The second outcome would demonstrate a potential limitation of time-shifted surrogates in event-related data.

# One important nuance: because theta phase is reset across trials, the gamma burst is consistently associated with particular theta phases. Thus, this simulation can contain measurable statistical phase–amplitude dependence even though there is no direct physiological interaction between the oscillations. Surrogates cannot necessarily distinguish a direct interaction from a common event-related cause.

# Also, a nonsignificant result would not prove that the procedure is immune to event-related confounds.

# For now, let's run this single test. **Please share the original PAC, surrogate mean, surrogate SD, and empirical p-value.**

# Then we'll decide whether any additional validation is needed before scaling up to Study 1.

Event-locked data shape: (84, 249)

EVENT-LOCKED SIGNAL
Original PAC: 0.0020461358176255976
Mean surrogate: 0.0019714434676718568
Surrogate SD: 6.500923480792277e-05
Empirical p: 0.1


**Conclusion:**
Our implementation has now passed three initial sanity checks:
1. It did not detect significant PAC in an uncoupled signal.
2. It detected strong, deliberately introduced PAC.
3. It did not detect significant PAC in one event-locked simulation without direct coupling.